# Bygga en intäktssäkringskub för telekom med PROC SUMMARY

## Sammanfattning

Ett intäktssäkringsteam hos en mobiloperatör förbehandlar en månads abonnentdagliga faktureringsposter till en kompakt sammanfattningskub, så att analytiker kan borra ner i reglerad intäkt per region, abonnemangsnivå och samtalstyp utan att skanna om den råa faktatabellen. `PROC SUMMARY` rullar upp 100 abonnentdagsposter till en flerdimensionell uppsättning celler: den finaste detaljnivån (region x abonnemangsnivå x samtalstyp) fyller 25 av 27 möjliga celler, medan namngivna delkuber ger de marginaler analytiker frågar efter mest. Under denna exempelmånad reglerade operatören \$222.78 över de tre aktiva regionerna, där Söder (\$97.44) och Öster (\$86.94) tillsammans står för 83% av intäkten och Norr släpar efter med \$38.40. Den mest lönsamma enskilda delkuben är Plus-nivåns röst-tjänst (\$59.06 över 18 poster), och att rangordna celler efter intäkt per post lyfter fram dataanvändningsceller som de mest värdefulla målen för en läckagerevision (högsta utbyte \$7.87/post). Varje siffra nedan läses direkt från den körda utmatningen.

## Datakällor

| Dataset | Rader | Detaljnivå | Nyckelvariabler |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | En rad per abonnentdags användningssammanfattning | `region` (Öster/Söder/Norr), `plan_tier` (Kontantkort/Standard/Plus), `call_type` (Röst/SMS/Data), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | En rad per icke-tom (region x plan_tier x call_type)-cell | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | En rad per cell i de namngivna borrningsdelkuberna | `_TYPE_`, `_FREQ_`, `rev_total` |

All data genereras direkt i koden med `call streaminit()` / `rand()`; inga externa filer eller nätverksåtkomst krävs. Den här miljön körs olicensierad, så skrivna dataset begränsas till 100 observationer - scenariot är dimensionerat så att kuben blir fullständigt fylld inom den gränsen.

## Från råa samtalsdetaljposter till en borrningsbar kub

Mobiloperatörer reglerar intäkter över miljontals dagliga användningshändelser. För att låta intäktssäkringsanalytiker svara på frågor som *"Vad var Plus-nivåns röstintäkt i regionen Söder förra månaden?"* utan att skanna om den råa faktatabellen varje gång, **förbehandlar** vi data till en kompakt sammanfattningskub.

`PROC SUMMARY` är Base SAS arbetshäst för precis detta: den grupperar en platt faktatabell efter en eller flera `CLASS`-dimensioner och skriver de begärda statistiken till ett utdataset, och märker varje rad med `_TYPE_` (vilka dimensioner som är aktiva) och `_FREQ_` (poster bakom cellen). Det utdatasetet *är* en flerdimensionell kub - samma upprullningsstruktur som ett OLAP-verktyg skulle exponera, materialiserad som ett vanligt SAS-dataset du kan skriva ut, sammanfoga och skiva.

Den här notebooken:

1. Genererar en realistisk månads abonnentdagliga faktureringsposter.
2. Bygger den finaste detaljnivåns kub (region x abonnemangsnivå x samtalstyp) med `PROC SUMMARY NWAY`.
3. Materialiserar namngivna **borrningsdelkuber** med `TYPES`-satsen.
4. Projicerar intäkt till abonnentbasen med en `WEIGHT`, borrar ner i en region, och rangordnar celler efter intäkt per post för läckageanalys.

## Steg 1 - Generera syntetisk samtalsdetalj-/faktureringsdata

Varje rad sammanfattar en abonnents användning av en tjänst under en dag. Vi använder `call streaminit` för reproducerbarhet och `rand()` för att dra rimliga fördelningar: intäkt och användning skalas med abonnemangsnivån, röstintäkt följer fakturerbara minuter, och dataintäkt följer megabyte. Varje `RAND('table', ...)` ges en sannolikhet per kategori så att varje region, nivå och samtalstyp förekommer i urvalet på 100 poster. En liten `subscriber_wt`-undersökningsvikt är knuten så att vi senare kan visa ett viktat mått.

In [1]:
data work.cdr_billing;
    CALL streaminit(20260131);
    LÄNGD region $10 plan_tier $16 call_type $10 device_class $12 bill_month $7;
    ETIKETT region        = "Region"
          plan_tier     = "Abonnemangsnivå"
          call_type     = "Samtalstyp"
          device_class  = "Enhetstyp"
          bill_month    = "Faktureringsmånad"
          revenue       = "Reglerad intäkt (USD)"
          call_minutes  = "Fakturerbara samtalsminuter"
          data_mb       = "Datavolym (MB)"
          subscriber_wt = "Abonnentens undersökningsvikt";

    GÖR i = 1 TILL 100;
        /* --- Dimensioner: en sannolikhet per nivå, summerar till 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        VÄLJ (r);
            NÄR (1) region = "Öster";
            NÄR (2) region = "Söder";
            ANNARS_OM region = "Norr";
        SLUT;

        p = rand("table", 0.30, 0.40, 0.30);
        VÄLJ (p);
            NÄR (1) plan_tier = "Kontantkort";
            NÄR (2) plan_tier = "Standard";
            ANNARS_OM plan_tier = "Plus";
        SLUT;

        c = rand("table", 0.50, 0.30, 0.20);
        VÄLJ (c);
            NÄR (1) call_type = "Röst";
            NÄR (2) call_type = "SMS";
            ANNARS_OM call_type = "Data";
        SLUT;

        OM rand("uniform") < 0.55 SÅ device_class = "Smart";
        ANNARS device_class = "Enkel";

        bill_month = "2026-01";

        /* --- Mått, skalade efter nivå och tjänst --- */
        tier_mult = (plan_tier = "Kontantkort")*0.7
                  + (plan_tier = "Standard")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Röst")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Data")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        UTDATA;
    SLUT;
    TA_BORT i r p c tier_mult base_rev;
KÖR;

PROC PRINT data=work.cdr_billing(obs=8) ETIKETT noobs;
    TITEL "Provexempel: abonnentdagens faktureringsposter";
KÖR;


                                     Provexempel: abonnentdagens faktureringsposter                                     

Region   Abonnemangsnivå  Samtalstyp  Enhetstyp   Faktureringsmånad  Fakturerbara samtalsminuter  Datavolym (MB)   Reglerad intäkt (USD)   Abonnentens undersökningsvikt
Norr    Plus              SMS         Smart      2026-01                                       0               0                    0.67                            1.13
Söder   Kontantkort       SMS         Enkel      2026-01                                       0               0                    0.94                            1.42
Söder   Plus              SMS         Smart      2026-01                                       0               0                    0.99                            0.86
Söder   Plus              SMS         Smart      2026-01                                       0               0                    1.01                            1.03
Söder   Plus              Röst   


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Steg 2 - Bygg den finaste detaljnivåns kub med PROC SUMMARY NWAY

`NWAY` behåller endast den enda mest detaljerade kombinationen av alla `CLASS`-dimensioner - här varje befolkad (region x plan_tier x call_type)-cell. För varje cell lagrar vi intäktens `SUM`, `MEAN` och `MAX` plus totala minuter och megabyte, så att en analytiker kan läsa total intäkt per cell, härleda genomsnittet per post, och upptäcka den största enskilda debiteringen. `_FREQ_` anger hur många abonnentdagar som ligger bakom varje cell. Vi tar bort `_TYPE_` här eftersom, på NWAY-nivån, har varje rad samma typ.

In [2]:
PROC SUMMARY data=work.cdr_billing NWAY;
    KLASS region plan_tier call_type;
    VARIABEL revenue call_minutes data_mb;
    UTDATA out=work.cube_nway(drop=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
KÖR;

PROC PRINT data=work.cube_nway(obs=12) noobs ETIKETT;
    ETIKETT region="Region" plan_tier="Abonnemangsnivå" call_type="Samtalstyp"
          _freq_="Antal poster"
          rev_total="Total intäkt" rev_mean="Genomsnittlig intäkt" rev_max="Max intäkt"
          min_total="Totalt antal minuter" data_total="Total datavolym";
    TITEL "NWAY-kubceller (region * plan_tier * call_type)";
    format rev_total rev_mean rev_max min_total data_total comma10.2;
KÖR;


                                    NWAY-kubceller (region * plan_tier * call_type)                                     

Region   Abonnemangsnivå  Samtalstyp  Antal poster   Total intäkt   Genomsnittlig intäkt   Max intäkt  Totalt antal minuter  Total datavolym
Norr    Kontantkort       Data                   2           2.16                   1.08         1.09                  0.00           229.50
Norr    Kontantkort       Röst                   5           8.56                   1.71         2.15                165.70             0.00
Norr    Kontantkort       SMS                    2           2.00                   1.00         1.18                  0.00             0.00
Norr    Plus              Röst                   3           8.12                   2.71         3.80                149.00             0.00
Norr    Plus              SMS                    4           3.00                   0.75         0.97                  0.00             0.00
Norr    Standard          Data  


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Steg 3 - Materialisera namngivna borrningsdelkuber med TYPES

En OLAP-kub förlagrar de upprullningar analytiker navigerar mest. `TYPES`-satsen gör precis det: varje term ber `PROC SUMMARY` att generera en delkub. Vi begär totalsumman `()`, `region`-marginalen, och de tvåvägs delkuberna `region*plan_tier` och `call_type*plan_tier` - de borrningsvägar en intäktspanel skulle exponera. SAS märker varje delkub med en `_TYPE_`-kod (en bitmask över `CLASS`-listan), så ett enda utdataset bär varje nivå av hierarkin.

In [3]:
PROC SUMMARY data=work.cdr_billing;
    KLASS region plan_tier call_type;
    VARIABEL revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    UTDATA out=work.cube_hier
           sum(revenue)=rev_total;
KÖR;

PROC PRINT data=work.cube_hier noobs ETIKETT;
    ETIKETT _type_="Typkod" region="Region" plan_tier="Abonnemangsnivå" call_type="Samtalstyp"
          _freq_="Antal poster" rev_total="Total intäkt";
    TITEL "Delkubernas sammanställningar: totalsumma, region, region*nivå, samtalstyp*nivå";
    VARIABEL _type_ region plan_tier call_type _freq_ rev_total;
    format rev_total comma10.2;
KÖR;


                    Delkubernas sammanställningar: totalsumma, region, region*nivå, samtalstyp*nivå                     

Typkod  Region   Abonnemangsnivå  Samtalstyp  Antal poster   Total intäkt
     0                                                 100         222.78
     3          Kontantkort       Data                   7          14.50
     3          Kontantkort       Röst                  16          24.77
     3          Kontantkort       SMS                    7           6.82
     3          Plus              Data                   3          17.46
     3          Plus              Röst                  18          59.06
     3          Plus              SMS                   13          11.75
     3          Standard          Data                   8          38.06
     3          Standard          Röst                  20          42.33
     3          Standard          SMS                    8           8.03
     4  Norr                                            23      


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Steg 4 - Viktad projektion, en regional fördjupning, och läckageanalys

Tre avläsningar ett intäktssäkringsteam faktiskt gör mot kuben:

- **Viktad projektion.** Att knyta `WEIGHT=subscriber_wt` till en `region*plan_tier`-sammanfattning rapporterar intäkt skalad till hela den abonnentbas urvalet representerar, snarare än de råa samplade raderna.
- **Fördjupning.** Att filtrera NWAY-kuben till en region expanderar en enskild gren av hierarkin - här Söder - till dess nivå-per-tjänst-detalj.
- **Läckageanalys.** Att sortera cellerna efter genomsnittlig intäkt per post lyfter fram cellerna med högst utbyte; celler med låg frekvens och högt utbyte är exakt vad en revision granskar för feldebiterad eller läckande intäkt.

In [4]:
/* Viktad intäkt projicerad på abonnentbasen */
PROC SUMMARY data=work.cdr_billing NWAY;
    KLASS region plan_tier;
    VARIABEL revenue;
    VIKT subscriber_wt;
    UTDATA out=work.cube_wt(drop=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
KÖR;

PROC PRINT data=work.cube_wt noobs ETIKETT;
    ETIKETT region="Region" plan_tier="Abonnemangsnivå" rev_weighted="Viktad intäkt" records="Antal poster";
    TITEL "Viktad intäkt per region * abonnemangsnivå (projicerad på abonnentbasen)";
    format rev_weighted comma10.2;
KÖR;

/* Fördjupning i regionen Söder i kuben */
PROC PRINT data=work.cube_nway noobs ETIKETT;
    DÄR region = "Söder";
    ETIKETT plan_tier="Abonnemangsnivå" call_type="Samtalstyp" _freq_="Antal poster"
          rev_total="Total intäkt" rev_mean="Genomsnittlig intäkt";
    TITEL "Fördjupning i Söder: intäktsceller per nivå och samtalstyp";
    VARIABEL plan_tier call_type _freq_ rev_total rev_mean;
    format rev_total rev_mean comma10.2;
KÖR;

/* Rangordna celler efter intäkt per post för läckageanalys */
PROC SORT data=work.cube_nway out=work.cube_ranked;
    EFTER FALLANDE rev_mean;
KÖR;

PROC PRINT data=work.cube_ranked(obs=6) noobs ETIKETT;
    ETIKETT region="Region" plan_tier="Abonnemangsnivå" call_type="Samtalstyp" _freq_="Antal poster"
          rev_mean="Genomsnittlig intäkt" rev_max="Max intäkt";
    TITEL "Celler med högst genomsnittlig intäkt (utbyte per post)";
    VARIABEL region plan_tier call_type _freq_ rev_mean rev_max;
    format rev_mean rev_max comma10.2;
KÖR;


                        Viktad intäkt per region * abonnemangsnivå (projicerad på abonnentbasen)                        

Region   Abonnemangsnivå   Viktad intäkt  Antal poster
Norr    Kontantkort                20.56             9
Norr    Plus                       22.42             7
Norr    Standard                   18.30             7
Söder   Kontantkort                27.77            10
Söder   Plus                       56.29            15
Söder   Standard                   58.63            14
Öster   Kontantkort                29.77            11
Öster   Plus                       59.59            12
Öster   Standard                   50.85            15

                               Fördjupning i Söder: intäktsceller per nivå och samtalstyp                               

 Abonnemangsnivå  Samtalstyp  Antal poster   Total intäkt   Genomsnittlig intäkt
Kontantkort       Data                   5          12.34                   2.47
Kontantkort       Röst                   3 


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Tolkning av resultaten

Sammanfattningskuben omvandlar 100 råa abonnentdagsrader till en kompakt uppsättning förberäknade celler som besvarar fördjupningsfrågor omedelbart, utan att skanna om faktatabellen:

- **Var intäkten finns.** `region`-marginalen (`_TYPE_=4`) visar att Söder reglerade \$97.44 och Öster \$86.94 - tillsammans 83% av månadens \$222.78 - medan Norr bidrog med \$38.40. Inom delkuben `call_type*plan_tier` (`_TYPE_=3`) är Plus-nivåns röst den enskilt mest lönsamma cellen med \$59.06 över 18 poster, med Standard-nivåns röst därnäst på \$42.33.
- **Viktad projektion.** Att tillämpa undersökningsvikten omfördelar rangordningen mot abonnemang vars abonnenter bär mer vikt: Öster-Plus (\$59.59) och Söder-Standard (\$58.63) leder den projicerade `region*plan_tier`-intäkten, en annan bild än de oviktade regiontotalerna och en påminnelse om att rapportera projicerad snarare än samplad intäkt vid dimensionering av exponering.
- **Utbyte per post och läckageanalys.** Att rangordna NWAY-celler efter `rev_mean` placerar dataanvändningsceller överst - Norr-Standard-Data (\$7.87/post) och Söder-Plus-Data (\$5.96/post) - vilket bekräftar att datatung användning ger den högsta intäkten per post. Ledarna med låg frekvens (en eller två poster) är precis de celler en intäktssäkringsanalytiker skulle hämta de underliggande samtalsdetaljposterna för, för att bekräfta att den höga debiteringen är korrekt prissatt snarare än ett fel.

För ett intäktssäkringsteam är den här kuben grunden för avvikelsepaneler: jämför reglerad intäkt mot förväntad prislisteintäkt per (region, nivå, samtalstyp)-cell, och de celler vars medelvärde eller viktade totalsumma avviker mest blir de läckagefall som är värda att revidera. Eftersom hela strukturen är ett vanligt SAS-dataset kan nästa månads kub unioneras, differentieras, eller sammanfogas med en prislista med samma Base SAS-verktyg.